# Unit 2 Hands-On ①: FrozenLake-v1 Q-Learning 실습

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 2**의 첫 번째 실습입니다.  
**Q-Learning** 알고리즘으로 `FrozenLake-v1` 환경에서 에이전트를 훈련하고,  
훈련 과정을 영상으로 기록하여 Google Drive에 저장합니다.

---
## 목차
1. 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. 라이브러리 임포트
5. 환경 탐색
6. Q-테이블 초기화
7. 정책 함수 정의
8. 하이퍼파라미터 설정
9. 훈련 함수 정의 및 실행
10. 평가
11. 훈련 과정 영상 확인
12. Hugging Face Hub 업로드

---
## 1. 환경 설치

FrozenLake 실행에 필요한 패키지를 설치합니다.  
`pickle5`는 Python 3.9+ 에서 내장되어 있어 별도 설치가 불필요하므로 제외합니다.

In [1]:
# pickle5를 제외하고 필요한 패키지만 직접 설치
# (requirements-unit2.txt 의 pickle5, pyyaml==6.0 은 Python 3.12 에서 빌드 오류 발생)
!pip install gymnasium pygame numpy huggingface_hub imageio imageio-ffmpeg tqdm

In [2]:
# 가상 디스플레이용 패키지 설치 (Colab 에는 모니터가 없으므로 필요)
!sudo apt-get update -qq
!sudo apt-get install -y python3-opengl ffmpeg xvfb -qq
!pip install pyvirtualdisplay -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package freeglut3:amd64.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../freeglut3_2.8.1-6_amd64.deb ...
Unpacking freeglut3:amd64 (2.8.1-6) ...
Selecting previously unselected package libglu1-mesa:amd64.
Preparing to unpack .../libglu1-mesa_9.0.2-1_amd64.deb ...
Unpacking libglu1

---
## 2. Google Drive 마운트

Colab VM은 세션 종료 시 파일이 모두 삭제됩니다.  
Google Drive에 마운트하여 훈련 영상과 모델을 영구 보존합니다.

```
Google Drive/RL_Course/Unit2_FrozenLake/
├── training_videos/   ← 훈련 중간 단계별 영상
└── q-frozenlake.pkl   ← 최종 Q-테이블 모델
```

> 실행 시 Google 계정 인증 팝업이 뜹니다. 허용해주세요.

In [3]:
from google.colab import drive
import os

# Google Drive 마운트
drive.mount('/content/drive')

# ✏️ 저장 폴더명을 원하는 대로 변경하세요.
DRIVE_BASE  = "/content/drive/MyDrive/RL_Course/Unit2_FrozenLake"
VIDEO_DIR   = f"{DRIVE_BASE}/training_videos"
MODEL_DIR   = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("✅ Drive 마운트 완료!")
print(f"   영상 저장 경로 : {VIDEO_DIR}")
print(f"   모델 저장 경로 : {MODEL_DIR}")

Mounted at /content/drive
✅ Drive 마운트 완료!
   영상 저장 경로 : /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos
   모델 저장 경로 : /content/drive/MyDrive/RL_Course/Unit2_FrozenLake


---
## 3. 가상 디스플레이 설정

Colab에는 물리적인 화면이 없으므로, rgb_array 렌더링을 위해 가상 디스플레이를 생성합니다.

In [4]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print("✅ 가상 디스플레이 시작")

✅ 가상 디스플레이 시작


---
## 4. 라이브러리 임포트

| 라이브러리 | 역할 |
|---|---|
| `numpy` | Q-테이블 생성 및 수학 연산 |
| `gymnasium` | 강화학습 환경 |
| `random` | ε-greedy 탐색의 랜덤 행동 선택 |
| `imageio` | 프레임을 mp4로 저장 |
| `tqdm` | 훈련 진행률 표시 |
| `pickle` | Q-테이블 직렬화 저장 |

In [5]:
import numpy as np
import gymnasium as gym
import random
import imageio
import pickle
import glob

from tqdm.notebook import tqdm
from IPython.display import Video, display

---
## 5. 환경 탐색

### FrozenLake-v1 이란?

4×4 격자판 위에서 출발점(S)에서 목표(G)까지 이동하는 환경입니다.  
구멍(H)에 빠지면 에피소드가 종료됩니다.

```
S F F F      S: 출발점 (Start)
F H F H      F: 이동 가능 (Frozen)
F F F H      H: 구멍 (Hole) → 빠지면 실패
H F F G      G: 목표 (Goal)
```

`is_slippery=False`: 항상 의도한 방향으로 이동 (결정론적 환경)

In [6]:
# FrozenLake 환경 생성 (rgb_array 모드: 프레임을 배열로 받음)
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

print("===== 관측 공간(Observation Space) =====")
print("크기:", env.observation_space.n, "→ 4×4 격자의 16개 칸")
print("샘플:", env.observation_space.sample())

print("\n===== 행동 공간(Action Space) =====")
print("크기:", env.action_space.n, "→ 4방향 이동")
print("행동 의미: 0=왼쪽, 1=아래, 2=오른쪽, 3=위")

===== 관측 공간(Observation Space) =====
크기: 16 → 4×4 격자의 16개 칸
샘플: 14

===== 행동 공간(Action Space) =====
크기: 4 → 4방향 이동
행동 의미: 0=왼쪽, 1=아래, 2=오른쪽, 3=위


In [7]:
state_space  = env.observation_space.n  # 16
action_space = env.action_space.n       # 4

print(f"상태 공간 크기: {state_space}")
print(f"행동 공간 크기: {action_space}")

상태 공간 크기: 16
행동 공간 크기: 4


---
## 6. Q-테이블 초기화

**Q-테이블**: (상태 수 × 행동 수) 크기의 2D 배열로, 각 셀은 Q(s, a) 값을 저장합니다.  
처음에는 모든 값을 0으로 초기화하고, 훈련을 통해 점차 업데이트됩니다.

```
Q-테이블 (16×4)
       왼  아래  오른  위
상태0 [ 0,   0,   0,   0 ]
상태1 [ 0,   0,   0,   0 ]
  ...
상태15[ 0,   0,   0,   0 ]
```

In [8]:
def initialize_q_table(state_space, action_space):
    """모든 Q값을 0으로 초기화한 Q-테이블 생성"""
    return np.zeros((state_space, action_space))

Qtable_frozenlake = initialize_q_table(state_space, action_space)
print(f"Q-테이블 크기: {Qtable_frozenlake.shape}  (상태 {state_space} × 행동 {action_space})")
print(Qtable_frozenlake)

Q-테이블 크기: (16, 4)  (상태 16 × 행동 4)
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


---
## 7. 정책 함수 정의

### Greedy Policy (탐욕적 정책)
현재 Q-테이블에서 **가장 높은 Q값**을 가진 행동을 선택합니다.  
평가(테스트) 시에 사용합니다.

### ε-Greedy Policy (엡실론-탐욕적 정책)
훈련 초반에는 **탐색(Exploration)**을 많이 하고, 후반에는 **활용(Exploitation)**을 많이 합니다.

```
랜덤값 > ε  →  활용: Q-테이블에서 최선의 행동 선택
랜덤값 ≤ ε  →  탐색: 완전 랜덤 행동 선택
```

ε는 훈련이 진행될수록 지수적으로 감소합니다 (`decay_rate` 조절).

In [9]:
def greedy_policy(Qtable, state):
    """Q값이 최대인 행동 선택 (평가용)"""
    return np.argmax(Qtable[state][:])


def epsilon_greedy_policy(Qtable, state, epsilon):
    """ε 확률로 랜덤 탐색, 1-ε 확률로 탐욕적 선택 (훈련용)"""
    if random.uniform(0, 1) > epsilon:
        action = greedy_policy(Qtable, state)   # 활용
    else:
        action = env.action_space.sample()       # 탐색
    return action

---
## 8. 하이퍼파라미터 설정

| 파라미터 | 값 | 설명 |
|---|---|---|
| `n_training_episodes` | 10,000 | 총 훈련 에피소드 수 |
| `learning_rate` | 0.7 | 학습률 α: Q값 업데이트 보폭 |
| `n_eval_episodes` | 100 | 평가용 에피소드 수 |
| `max_steps` | 99 | 에피소드당 최대 스텝 |
| `gamma` | 0.95 | 할인율: 미래 보상 반영 비율 |
| `max_epsilon` | 1.0 | 초기 탐색률 (100% 탐색) |
| `min_epsilon` | 0.05 | 최소 탐색률 (5% 유지) |
| `decay_rate` | 0.0005 | ε 감소 속도 |
| `video_freq` | 2,000 | 몇 에피소드마다 영상 저장 |

In [10]:
# 훈련 파라미터
n_training_episodes = 10_000   # 총 훈련 에피소드
learning_rate       = 0.7      # 학습률 α

# 평가 파라미터
n_eval_episodes = 100          # 평가 에피소드 수

# 환경 파라미터
env_id    = "FrozenLake-v1"
max_steps = 99                 # 에피소드당 최대 스텝
gamma     = 0.95               # 할인율
eval_seed = []                 # 평가 시드 (FrozenLake는 빈 리스트)

# 탐색 파라미터
max_epsilon = 1.0              # 초기 탐색률
min_epsilon = 0.05             # 최소 탐색률
decay_rate  = 0.0005           # ε 감소 속도

# 영상 저장 주기
video_freq = 2_000             # ✏️ 몇 에피소드마다 영상을 저장할지

---
## 9. 훈련 함수 정의 및 실행

### Q-Learning 업데이트 공식

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ R + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

| 기호 | 의미 |
|---|---|
| $Q(s,a)$ | 현재 상태 s에서 행동 a를 했을 때의 Q값 |
| $\alpha$ | 학습률 (learning_rate) |
| $R$ | 즉각적인 보상 |
| $\gamma$ | 할인율 (gamma) |
| $\max Q(s',a')$ | 다음 상태 s'에서의 최대 Q값 |

### 영상 저장 방식
`video_freq` 에피소드마다 현재 Q-테이블로 1 에피소드를 실행하여 mp4로 저장합니다.  
훈련이 끝난 후 시간 순서대로 재생하면 학습 과정을 확인할 수 있습니다.

In [11]:
def save_episode_video(env, Qtable, video_path, fps=2):
    """현재 Q-테이블로 1 에피소드를 실행하여 mp4로 저장"""
    frames = []
    state, _ = env.reset(seed=random.randint(0, 500))
    frames.append(env.render())
    terminated = truncated = False

    while not (terminated or truncated):
        action = greedy_policy(Qtable, state)
        state, _, terminated, truncated, _ = env.step(action)
        frames.append(env.render())

    imageio.mimsave(video_path, [np.array(f) for f in frames], fps=fps)


def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate,
          env, max_steps, Qtable, video_freq, video_dir):
    """Q-Learning 훈련 루프. video_freq 에피소드마다 영상 저장."""

    for episode in tqdm(range(n_training_episodes), desc="훈련 진행"):

        # ε 지수 감소: 훈련 초반에는 탐색 많이, 후반에는 활용 많이
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

        state, _ = env.reset()
        terminated = truncated = False

        for step in range(max_steps):
            # ε-greedy 정책으로 행동 선택
            action = epsilon_greedy_policy(Qtable, state, epsilon)

            # 환경에 행동 적용 → 다음 상태, 보상 수신
            new_state, reward, terminated, truncated, _ = env.step(action)

            # Q-Learning 업데이트
            Qtable[state][action] += learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )

            if terminated or truncated:
                break

            state = new_state

        # video_freq 에피소드마다 영상 저장
        if (episode + 1) % video_freq == 0:
            video_path = f"{video_dir}/episode_{episode+1:06d}.mp4"
            save_episode_video(env, Qtable, video_path)
            print(f"   🎬 [{episode+1:,} 에피소드] 영상 저장 → {video_path}")

    return Qtable

In [12]:
# 훈련 실행
Qtable_frozenlake = train(
    n_training_episodes, min_epsilon, max_epsilon, decay_rate,
    env, max_steps, Qtable_frozenlake,
    video_freq=video_freq,
    video_dir=VIDEO_DIR
)

print("\n✅ 훈련 완료!")
print("\n최종 Q-테이블:")
print(Qtable_frozenlake)

훈련 진행:   0%|          | 0/10000 [00:00<?, ?it/s]

   🎬 [2,000 에피소드] 영상 저장 → /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_002000.mp4
   🎬 [4,000 에피소드] 영상 저장 → /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_004000.mp4
   🎬 [6,000 에피소드] 영상 저장 → /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_006000.mp4
   🎬 [8,000 에피소드] 영상 저장 → /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_008000.mp4
   🎬 [10,000 에피소드] 영상 저장 → /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_010000.mp4

✅ 훈련 완료!

최종 Q-테이블:
[[0.73509189 0.77378094 0.77378094 0.73509189]
 [0.73509189 0.         0.81450625 0.77378094]
 [0.77378094 0.857375   0.77378094 0.81450625]
 [0.81450625 0.         0.77378094 0.77378094]
 [0.77378094 0.81450625 0.         0.73509189]
 [0.         0.         0.         0.        ]
 [0.         0.9025     0.         0.81450625]
 [0.         0.         0.         0.        ]
 [0.81450625 0.         0.857375   0.77378094]


---
## 10. 평가

100번의 에피소드로 평균 보상을 측정합니다.  
- **보상 1.0**: 목표(G)에 도달 성공  
- **보상 0.0**: 구멍(H)에 빠지거나 시간 초과

평균 보상이 **0.9 이상**이면 잘 학습된 것입니다.

In [13]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
    """n_eval_episodes 동안 에이전트를 실행하여 평균 보상 반환"""
    episode_rewards = []

    for episode in tqdm(range(n_eval_episodes), desc="평가 진행"):
        state, _ = env.reset(seed=seed[episode]) if seed else env.reset()
        total_reward = 0.0
        terminated = truncated = False

        for _ in range(max_steps):
            action = greedy_policy(Q, state)  # 평가 시 항상 탐욕적 선택
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"\n평균 보상: {mean_reward:.2f} ± {std_reward:.2f}")
print("(1.0에 가까울수록 성공률이 높음)")

평가 진행:   0%|          | 0/100 [00:00<?, ?it/s]


평균 보상: 1.00 ± 0.00
(1.0에 가까울수록 성공률이 높음)


---
## 11. 훈련 과정 영상 확인

Drive에 저장된 영상을 순서대로 재생하여 학습 과정을 확인합니다.

In [14]:
# 저장된 영상 목록 출력
videos = sorted(glob.glob(f"{VIDEO_DIR}/*.mp4"))
print(f"총 {len(videos)}개의 영상이 저장되었습니다:\n")
for v in videos:
    ep = v.split("episode_")[1].split(".")[0]
    print(f"  {int(ep):,} 에피소드 시점: {v}")

총 5개의 영상이 저장되었습니다:

  2,000 에피소드 시점: /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_002000.mp4
  4,000 에피소드 시점: /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_004000.mp4
  6,000 에피소드 시점: /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_006000.mp4
  8,000 에피소드 시점: /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_008000.mp4
  10,000 에피소드 시점: /content/drive/MyDrive/RL_Course/Unit2_FrozenLake/training_videos/episode_010000.mp4


In [15]:
# 모든 영상을 순서대로 노트북에서 재생
for video_path in videos:
    ep = video_path.split("episode_")[1].split(".")[0]
    print(f"\n📽️  {int(ep):,} 에피소드 시점")
    display(Video(video_path, embed=True, width=400))


📽️  2,000 에피소드 시점



📽️  4,000 에피소드 시점



📽️  6,000 에피소드 시점



📽️  8,000 에피소드 시점



📽️  10,000 에피소드 시점


---
## 12. Hugging Face Hub 업로드

훈련된 Q-테이블을 HF Hub에 공유합니다.

### 사전 준비
1. [Hugging Face 계정 생성](https://huggingface.co/join)
2. [쓰기(write) 권한 토큰 발급](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰으로 교체하세요
# ⚠️ 토큰은 절대 외부에 공개하지 마세요!
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxx")

In [17]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from pathlib import Path
import datetime, json


def record_video(env, Qtable, out_directory, fps=2):
    """HF Hub용 최종 영상 1개 생성"""
    frames = []
    state, _ = env.reset(seed=random.randint(0, 500))
    frames.append(env.render())
    terminated = truncated = False
    while not (terminated or truncated):
        action = np.argmax(Qtable[state][:])
        state, _, terminated, truncated, _ = env.step(action)
        frames.append(env.render())
    imageio.mimsave(out_directory, [np.array(f) for f in frames], fps=fps)


def push_to_hub(repo_id, model, env, video_fps=2, local_repo_path="hub"):
    """평가 → 영상 생성 → HF Hub 업로드 전체 파이프라인"""
    _, repo_name = repo_id.split("/")
    api = HfApi()

    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Q-테이블 저장
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False
    with open(repo_local_path / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # 평가
    mean_reward, std_reward = evaluate_agent(
        env, model["max_steps"], model["n_eval_episodes"],
        model["qtable"], model["eval_seed"]
    )

    # 결과 JSON 저장
    with open(repo_local_path / "results.json", "w") as f:
        json.dump({"env_id": model["env_id"], "mean_reward": mean_reward,
                   "n_eval_episodes": model["n_eval_episodes"],
                   "eval_datetime": datetime.datetime.now().isoformat()}, f)

    # 모델 카드 메타데이터
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")
    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-no_slippery"

    metadata = {"tags": [env_name, "q-learning", "reinforcement-learning"]}
    eval_meta = metadata_eval_result(
        model_pretty_name=repo_name, task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning", metrics_pretty_name="mean_reward",
        metrics_id="mean_reward", metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name, dataset_id=env_name,
    )
    metadata = {**metadata, **eval_meta}

    readme_path = repo_local_path / "README.md"
    readme = readme_path.read_text(encoding="utf8") if readme_path.exists() else \
        f"# Q-Learning Agent playing {env_name}\n"
    readme_path.write_text(readme, encoding="utf-8")
    metadata_save(readme_path, metadata)

    # 영상 생성 및 업로드
    record_video(env, model["qtable"], repo_local_path / "replay.mp4", video_fps)
    api.upload_folder(repo_id=repo_id, folder_path=repo_local_path, path_in_repo=".")
    print("\n✅ 업로드 완료:", repo_url)

In [18]:
# 업로드할 모델 딕셔너리 구성
model = {
    "env_id":               env_id,
    "max_steps":            max_steps,
    "n_training_episodes":  n_training_episodes,
    "n_eval_episodes":      n_eval_episodes,
    "eval_seed":            eval_seed,
    "learning_rate":        learning_rate,
    "gamma":                gamma,
    "max_epsilon":          max_epsilon,
    "min_epsilon":          min_epsilon,
    "decay_rate":           decay_rate,
    "qtable":               Qtable_frozenlake,
}

In [19]:
# ✏️ 본인의 HF 사용자명과 저장소 이름을 입력하세요.
username  = "DitDahDitDit"                              # ← HF 사용자명 입력
repo_name = "q-FrozenLake-v1-4x4-noSlippery"

push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

평가 진행:   0%|          | 0/100 [00:00<?, ?it/s]


✅ 업로드 완료: https://huggingface.co/DitDahDitDit/q-FrozenLake-v1-4x4-noSlippery
